In [0]:
%run /Workspace/weather_notebook/nb_utils_dev

In [0]:
# ── Cell 2: Read Bronze ───────────────────────────────────────
log_header("air_quality_readings")
print("\n[1/4] Reading Bronze air quality JSON...")

json_path = list_bronze_json_paths("airquality")
df_raw    = spark.read.option("multiline", "true").json(json_path)
print(f"  Raw rows: {df_raw.count():,}")

In [0]:
# ── Cell 3: Transform ─────────────────────────────────────────
print("\n[2/4] Transforming...")

silver_airquality = (
    df_raw
    .withColumn("record",    F.explode(F.col("list")))
    .withColumn("latitude",  F.col("coord.lat").cast("double"))
    .withColumn("longitude", F.col("coord.lon").cast("double"))
    .withColumn("aqi",       F.col("record.main.aqi").cast("integer"))
    .withColumn("aqi_label",
        F.when(F.col("aqi") == 1, "Good")
         .when(F.col("aqi") == 2, "Fair")
         .when(F.col("aqi") == 3, "Moderate")
         .when(F.col("aqi") == 4, "Poor")
         .otherwise("Very Poor"))
    .withColumn("co",    F.col("record.components.co").cast("double"))
    .withColumn("no",    F.col("record.components.no").cast("double"))
    .withColumn("no2",   F.col("record.components.no2").cast("double"))
    .withColumn("o3",    F.col("record.components.o3").cast("double"))
    .withColumn("so2",   F.col("record.components.so2").cast("double"))
    .withColumn("pm2_5", F.col("record.components.pm2_5").cast("double"))
    .withColumn("pm10",  F.col("record.components.pm10").cast("double"))
    .withColumn("nh3",   F.col("record.components.nh3").cast("double"))
    .withColumn("reading_timestamp",
        F.from_unixtime(F.col("record.dt")).cast("timestamp"))
    .withColumn("reading_date", F.to_date(F.col("reading_timestamp")))
    .withColumn("reading_hour", F.hour(F.col("reading_timestamp")))
    .withColumn("pm2_5_who_status",
        F.when(F.col("pm2_5") <= 15,  "safe")
         .when(F.col("pm2_5") <= 45,  "moderate")
         .otherwise("unsafe"))
    .withColumn("pm10_who_status",
        F.when(F.col("pm10") <= 45,   "safe")
         .when(F.col("pm10") <= 100,  "moderate")
         .otherwise("unsafe"))
    .withColumn("no2_who_status",
        F.when(F.col("no2") <= 25,    "safe")
         .when(F.col("no2") <= 200,   "moderate")
         .otherwise("unsafe"))
    .withColumn("ingestion_date",  F.current_date())
    .withColumn("ingestion_ts",    F.current_timestamp())
    .withColumn("source_system",   F.lit("openweathermap_api"))
    .withColumn("source_endpoint", F.lit("/data/2.5/air_pollution"))
    .drop("list", "record", "coord")
    .filter(F.col("aqi").between(1, 5))
    .filter(F.col("reading_timestamp").isNotNull())
)

row_count = silver_airquality.count()
print(f"  Rows: {row_count:,}")

In [0]:
# ── Cell 4: Write to ADLS Gen2 + register in Unity Catalog ───
print("\n[3/4] Writing to ADLS Gen2...")
row_count = write_silver_table(silver_airquality, "air_quality_readings")

In [0]:
# ── Cell 5: Verify ────────────────────────────────────────────
print("\n[4/4] Verifying...")

df_verify = spark.table(f"{silver_catalog}.air_quality_readings")
total = df_verify.count()

print(f"  Total rows: {total:,}")

print("\nAQI distribution:")
spark.sql(f"""
    SELECT aqi_label,
           COUNT(*) as readings,
           ROUND(AVG(pm2_5), 2) as avg_pm2_5,
           ROUND(AVG(no2), 2)   as avg_no2,
           ROUND(AVG(o3), 2)    as avg_o3
    FROM {silver_catalog}.air_quality_readings
    GROUP BY aqi_label
    ORDER BY aqi_label
""").show()

print("\nLatest reading per location:")
spark.sql(f"""
    SELECT latitude, longitude, aqi_label, pm2_5, no2,
           reading_timestamp, reading_date
    FROM {silver_catalog}.air_quality_readings
    ORDER BY reading_timestamp DESC
    LIMIT 10
""").show(truncate=False)

status = "PASS"
log_footer("air_quality_readings", total, status)
dbutils.notebook.exit(f"air_quality_readings|{total}|{status}")